In [1]:
import pandas as pd
import numpy as np
from RuleTree import RuleTreeClassifier
from HybridReaders import read_wdbc
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier as knn
from sklearn.linear_model import LogisticRegression as lr
from sklearn.metrics import accuracy_score, classification_report, f1_score, jaccard_score, recall_score, precision_score
from sklearn.preprocessing import StandardScaler
from scipy.stats import kendalltau, spearmanr, pearsonr
import random as rd
from joblib import dump, load
import shap
import os

C:\Users\franc\anaconda3\envs\py3.12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Dataframe splitting
It splits the df and optionally saves splits. The seed is fixed at 42 for reproducibility on the same df.   
It also optionally scales X

save -> True if we want to save the splits as csv   
name_file -> to change the name of the file depending on the df we are using as input

In [2]:
def split_ts(df_name, df, save = True, scale = True, test_size = 0.3, save_path = './'):
    
    y = df['y'].values
    X = df[df.columns[:-1]].values
    if scale:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
    
    X_tr, X_ts, y_tr, y_ts = train_test_split(X, y, test_size = test_size, random_state = 42, stratify = y)
    to_save = {f'{df_name}_X_tr': X_tr, f'{df_name}_X_ts': X_ts, f'{df_name}_y_tr': y_tr, f'{df_name}_y_ts': y_ts}
    
    if save:
        for key, el in to_save.items():
            try:
                pd.DataFrame(el).to_csv(f'{save_path}/{key}.csv', index=False)
            except FileNotFoundError:
                print('The directory does not exist or it is wrong')
        return X_tr, X_ts, y_tr, y_ts
    else:
        return X_tr, X_ts, y_tr, y_ts

if __name__ == '__main__':
    df_name, df = read_wdbc(basepath = "C:/Users/franc/OneDrive/Desktop/Magistrale/Tesi modelli equivalenti/")
    X_tr, X_ts, y_tr, y_ts = split_ts(df_name = df_name, df = df, save = False, save_path = 'C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Split_salvati')

### Model training
It trains n. models, saves them on a joblib file and prints accuracy and varying parameters just to give a first look

model_type -> the model we want to train (rtc, knn, logreg)
max_models -> n. of models we want to train   
min_d -> min depth for rtc   
max_d -> max depth for rtc

max_nb/ min_nb -> max/min neighbours



In [39]:
def train_models(model_type, df_name, X_tr, X_ts, y_tr, y_ts, max_models = 5, min_d = 3, max_d = 20, save_path = './', max_nb = 5, min_nb = 1):

    np.random.seed(0)
    n_neighbors = np.random.choice(np.arange(min_nb, max_nb +1), size = max_models, replace = False)
    C = np.random.choice(np.logspace(-4, 4), size = max_models, replace = False)
    for i in range(max_models):
        rd.seed(i)
        if model_type == 'rtc':
            model = RuleTreeClassifier(
                max_depth = rd.randint(min_d, max_d),
                criterion = rd.choice(('gini', 'entropy')),
                prune_useless_leaves=True
            )
            params = ['max_depth', 'criterion']
            
        elif model_type == 'knn':
            model = knn(
                n_neighbors = n_neighbors[i],
                weights = rd.choice(('uniform','distance'))
            )
            params = ['n_neighbors', 'weights']

        elif model_type == 'lr':
            model = lr(
                C = C[i],
                max_iter = 500
            )
            params = ['C']

        model.fit(X_tr, y_tr)
        try:
            dump(model, f'{save_path}/{df_name}_{model_type}_{i+1}.joblib')
        
        except FileNotFoundError:
            print('The directory does not exist or it is wrong')
 
        #solo per dare una prima occhiata veloce
        y_pred = model.predict(X_ts)
        print(f'{df_name}_{model_type}_{i+1} \naccuracy: {accuracy_score(y_ts, y_pred)}') 
        for par in params:
            print(f'{par}: {model.get_params()[par]}')
            
if __name__ == '__main__':
    df_name, df = read_wdbc(basepath = "C:/Users/franc/OneDrive/Desktop/Magistrale/Tesi modelli equivalenti/")
    X_tr, X_ts, y_tr, y_ts = split_ts(df_name = df_name, df = df, save = False, save_path = 'C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Split_salvati')
    train_models(model_type = 'knn', df_name = 'wdbc', X_tr = X_tr, X_ts = X_ts, y_tr = y_tr, y_ts = y_ts, save_path = 'C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc')
    
    

wdbc_knn_1 
accuracy: 0.9590643274853801
n_neighbors: 3
weights: distance
wdbc_knn_2 
accuracy: 0.9415204678362573
n_neighbors: 1
weights: uniform
wdbc_knn_3 
accuracy: 0.935672514619883
n_neighbors: 2
weights: uniform
wdbc_knn_4 
accuracy: 0.9590643274853801
n_neighbors: 4
weights: uniform
wdbc_knn_5 
accuracy: 0.9649122807017544
n_neighbors: 5
weights: uniform


### Collecting models performances and predictions

In [80]:
def create_df_performances(df_name, X_ts, y_ts, path = './', save = True):
    
    df = pd.DataFrame()
    for file_name in os.listdir(path):
        
        if file_name.endswith('.joblib'):
            
            model = load(os.join(path, file_name))

            perf_dict = {
                'model': file_name.split('.')[0],
                'y_pred': model.predict(X_ts),
                'pred_proba': model.predict_proba(X_ts),
                'y_real': y_ts,
                'f1_score': f1_score(y_ts, y_pred, average = 'weighted'),
                'me': 1 - accuracy_score(y_ts, y_pred)
            }
            
            df = pd.concat([df, pd.DataFrame([perf_dict])], ignore_index=True)

    if save:
        
        df.to_csv(f'{path}/{df_name}_perf_preds.csv', index = False)
        return df

    return df
        


### Measures and Functions
le dividiamo in inter/intra family   

**preds_concordance()**   (inter-family)   
prende due modelli, usa la jaccard per definire la concordanza con la ground truth e la concordanza tra loro. in base alla misura scelta dice se c'è concordanza tra i modelli o no. Ci rende un dizionario con le info 

measures: jac_diff, concordance   
parameters:   
measure -> per scegliere quale misura delle due usare   
diff_th -> threshold nel caso di jac_diff   
conc_th -> threshold per la concordance   
average -> uso la jaccard_score di sklearn. Average mi serve per calcolare la jaccard in base al tipo di classificazione



In [56]:
#A PRESCINDERE DALLA FAMIGLIA
#quanto le predizioni sono concordi a ground truth? uno è più concorde dell'altro?
#quanto sono concordi tra due modelli m1 e m2?
def preds_concordance(m1, m2, X_ts, y_ts, average = 'binary', measure = 'jac_diff', diff_th = 0.2, conc_th = 0.7):

    y1_pred = m1.predict(X_ts)
    y2_pred = m2.predict(X_ts)

    truth_d1 = jaccard_score(y_ts, y1_pred, average = average) #quanto m1 dista dalla ground truth?
    truth_d2 = jaccard_score(y_ts, y2_pred, average = average) #quanto m2 dista dalla ground truth?
    
    #due misure diverse per la concordanza
    jac_diff = abs(truth_d1 - truth_d2) #diff di concordanza con la gt
    concordance = jaccard_score(y1_pred, y2_pred, average = average) #quanto concordano tra loro i modelli?
    
    sim_dict = {
        'm1_distance_from_truth': truth_d1, 
        'm2_distance_from_truth': truth_d2, 
    }

    if measure == 'jac_diff':
            sim_dict['jac_diff'] = jac_diff
            sim_dict['concordant'] = jac_diff <= diff_th
    else:
            sim_dict['concordance'] = concordance
            sim_dict['concordant'] = concordance >= conc_th
    
    return sim_dict

 
if __name__ == '__main__':
    m1 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_knn_2.joblib')
    m2 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_rtc_1.joblib')
    
    d = preds_concordance(m1, m2, X_ts, y_ts)
    for key, el in d.items():
        print(f'{key}: {el}')
    

m1_distance_from_truth: 0.8529411764705882
m2_distance_from_truth: 0.7746478873239436
jac_diff: 0.07829328914664457
concordant: True


**prediction_confidence** (inter-family)   
prende due modelli, ne considera le previsioni che combaciano e studia se coincidono abbastanza per quanto riguarda la % di confidence   

misure:   
coeff di correlazione (per binary) -> spearman, pearson, kendall a scelta    
cueff di correlazione medio tra classi -> stesse metriche a scelta   

parametri:   
metric -> scegliamo tra le 3 indicate   
tolerance -> livello di tolleranza sul valore del coeff di corr per determinare se la sicurezza nella predizioni è simile

In [77]:
#A PRESCINDERE DALLA FAMIGLIA
#due modelli a parità di previsione, quanto variano nella confidence?
#in caso di predizioni opposte, quanto si discostano in confidence? sono entrambi abbastanza incerti o uno è marcatamente più sicuro? 
#ANCORA DA CAPIRE COME STRUTTURARE L'ULTIMO QUESITO IN CASO DI MULTICLASSE

def prediction_confidence(m1, m2, y_ts, X_ts, metric = 'spearman', tolerance = 0.7):

    y1_pred = m1.predict(X_ts)
    y2_pred = m2.predict(X_ts)

    #solo le prob in cui le predizioni coincidono
    y1_prob = m1.predict_proba(X_ts)[y1_pred == y2_pred]
    y2_prob = m2.predict_proba(X_ts)[y1_pred == y2_pred]

    #in caso di classificazione binaria
    if y1_prob.shape[1] == 2:
        
        if metric == 'spearman':
            conf_corr, _ = spearmanr(y1_prob[:, 1], y2_prob[:, 1])
        elif metric == 'pearson':
            conf_corr, _ = pearsonr(y1_prob[:, 1], y2_prob[:, 1])
        elif metric == 'kendall':
            conf_corr, _ = kendalltau(y1_prob[:, 1], y2_prob[:, 1])
        
        return conf_corr, conf_corr >= tolerance

    #in caso multiclasse: facciamo una media delle correlazioni
    else:
        corrs = []
        for cl in range(y1_prob.shape[1]):
            if metric == 'spearman':
                confidence_correlation, _ = spearmanr(y1_prob[:, cl], y2_prob[:, cl])
            elif metric == 'pearson':
                confidence_correlation, _ = pearsonr(y1_prob[:, cl], y2_prob[:, cl])
            elif metric == 'kendall':
                confidence_correlation, _ = kendalltau(y1_prob[:, cl], y2_prob[:, cl])
            
            corrs.append(conf_corr)
        
        return np.mean(corrs), conf_corr >= tolerance
            
if __name__ == '__main__':
    m1 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_knn_3.joblib')
    m2 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_lr_1.joblib')

    conf_corr, similar_confidence = prediction_confidence(m1, m2, y_ts, X_ts, metric = 'kendall')
    print(f'confidence_correlation: {conf_corr} \nsimilar_confidence: {similar_confidence}')

confidence_correlation: 0.5560078877678701 
similar_confidence: False


**class_representation** (inter-family)   
prende i modelli e ne calcola le differenze tra recall per ogni classe, poi fa una media

misure:   
repr_diff -> calcola la differenza tra recall per ogni classe   
mean_repr_diff -> media delle deifferenze in recall nelle classi

parametri:   
tolerance -> livello di errore (mean_repr_diff) sopra il quale non c'è similarità nella rappresentazione delle classi

In [103]:
#A PRESCINDERE DALLA FAMIGLIA
#considerando una certa proporzione di rappresentazione per ogni classe, quanto viene considerata ogni classe?

def class_representation(m1, m2, X_ts, y_ts, tolerance = 0.09):

    y1_pred = m1.predict(X_ts)
    y2_pred = m2.predict(X_ts)

    #quanti esempi di ogni classe sono stati catturati dai modelli
    rec1 = recall_score(y_ts, y1_pred, average = None)
    rec2 = recall_score(y_ts, y2_pred, average = None)

    repr_diff = rec1 - rec2 #differenza nelle recall per classe
    classes = sorted(list(set(y1_pred)))

    repr_dict = dict(zip(classes, repr_diff))
    mean_repr_diff = np.mean(list(repr_dict.values())) #differenza media di tutte le classi

    info_d = {
        'repr_differences': repr_dict,
        'mean_repr_diff': mean_repr_diff,
        'repr_similarity': mean_repr_diff <= tolerance
    }
    return info_d

if __name__ == '__main__':
    m1 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_knn_5.joblib')
    m2 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_rtc_1.joblib')

    info_d = class_representation(m1, m2, X_ts, y_ts)
    for key, val in info_d.items():
        print(f'{key}: {val}')
        

repr_differences: {0.0: 0.06542056074766356, 1.0: 0.046875}
mean_repr_diff: 0.05614778037383178
repr_similarity: True


**feature_influence()** (inter-family)   

misure:   
infl_corr -> calcola il coeff di correlazione (spearman, kendall, pearson) tra le shaplets dei modelli per ogni feature

parametri:   
metric -> pearson, spearman, kendall (scelta del tipo di coefficiente per la correlazione)
concord_preds -> true se si vuole lavorare sulle predizioni concordi, false se si lavora sulle discordi



In [ ]:
#A PRESCINDERE DALLA FAMIGLIA
#quando due modelli concordano, quale feature li influenza di più? C'è un perché diverso?
#oppure, quando concordano, c'è una correlazione nell'influenza delle features? (shapelets) 
#stessa cosa quando non concordano

def feature_influence(m1, m2, df, X_tr, X_ts, y_ts, concord_preds = True, metric = 'pearson'):
    
    y1_pred = m1.predict(X_ts)
    y2_pred = m2.predict(X_ts)

    #dipende se vogliamo le predizioni concordanti o discordanti (conc_preds)
    if concord_preds:
        idx = np.where(y1_pred == y2_pred)[0]
    else:
        idx = np.where(y1_pred != y2_pred)[0]

    expl1 = shap.Explainer(m1.predict, X_tr, feature_names = df.columns)
    expl2 = shap.Explainer(m2.predict, X_tr, feature_names = df.columns)

    #calcoliamo le shapelets solo per le instances che interessano a noi
    shap1_vals = expl1(X_ts[idx]).values
    shap2_vals = expl2(X_ts[idx]).values

    feat_names = expl1(X_ts[idx]).feature_names

    shap1_vals_d = {}
    shap2_vals_d = {}
    infl_corr_d = {}
    for ft in range(shap1_vals.shape[1]):
        
        shap1_vals_d[feat_names[ft]] = shap1_vals[:, ft] #dizionario shapelets m1 con nomi features
        shap2_vals_d[feat_names[ft]] = shap2_vals[:, ft]
        
        if metric == 'spearman':
            corr, _ = spearmanr(shap1_vals[:, ft], shap2_vals[:, ft])
        elif metric == 'pearson':
            infl_corr, _ = pearsonr(shap1_vals[:, ft], shap2_vals[:, ft])
        elif metric == 'kendall':
            infl_corr, _ = kendalltau(shap1_vals[:, ft], shap2_vals[:, ft])

        infl_corr_d[feat_names[ft]] = infl_corr #dizionario delle correlazioni per feature

    
    return shap1_vals_d, shap2_vals_d, infl_corr_d

        
        
    

        
    

    

In [3]:
m1 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_knn_5.joblib')
m2 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_rtc_1.joblib')

In [6]:
explainer = shap.Explainer(m1.predict, X_tr)
shap_vals = explainer(X_ts)

PermutationExplainer explainer: 172it [01:11,  2.40it/s]                         


In [16]:
shap_vals.feature_names

['Feature 0',
 'Feature 1',
 'Feature 2',
 'Feature 3',
 'Feature 4',
 'Feature 5',
 'Feature 6',
 'Feature 7',
 'Feature 8',
 'Feature 9',
 'Feature 10',
 'Feature 11',
 'Feature 12',
 'Feature 13',
 'Feature 14',
 'Feature 15',
 'Feature 16',
 'Feature 17',
 'Feature 18',
 'Feature 19',
 'Feature 20',
 'Feature 21',
 'Feature 22',
 'Feature 23',
 'Feature 24',
 'Feature 25',
 'Feature 26',
 'Feature 27',
 'Feature 28',
 'Feature 29']